## Expression prediction on our data from the lab

After a sanity check with scGPT, we confirmed it's prediction capabilities are much stronger than baseline using a revised pipeline. In this notebook we verift the transferability of that pipeline to our dataset, investigating differences in data that could cause mismatches.

### 1. Load sample data and verify representation

In [1]:
from pathlib import Path
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmread

ROOT = Path.home() / "scgpt-cna"
MODEL_DIR = ROOT / "models/whole-human"

CASE_ID = "C3N-00148"
SAMPLE_ID = "C3N-00148-01"

SAMPLE_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/"
    "benchmarking_spatial_scale/GDC-data"
) / CASE_ID / SAMPLE_ID

RNA_DIR = (
    SAMPLE_DIR / "single_cell"
    / "dbe28c52-cdbf-477e-a728-9a61fd4a8139"
)
DATA_DIR = RNA_DIR / "qc_filtered_bc_feature_matrix"
COPYKAT_FILE = RNA_DIR / "copykatOUT/_copykat_prediction.txt"

print("Python:", sys.executable)
assert Path(sys.prefix).name == ".venv-scgpt", "Select the scGPT kernel."

# Accept either compressed or uncompressed 10x files.
def locate_file(directory, name):
    candidates = [
        p for p in [directory / name, directory / (name + ".gz")]
        if p.is_file()
    ]
    assert len(candidates) == 1, (
        f"Expected exactly one {name} or {name}.gz in {directory}; "
        f"found {candidates}"
    )
    return candidates[0]

MATRIX_FILE = locate_file(DATA_DIR, "matrix.mtx")
FEATURE_FILE = locate_file(DATA_DIR, "features.tsv")
BARCODE_FILE = locate_file(DATA_DIR, "barcodes.tsv")

assert COPYKAT_FILE.is_file(), COPYKAT_FILE
assert (MODEL_DIR / "vocab.json").is_file()

print("Sample:", SAMPLE_ID)
print("Matrix:", MATRIX_FILE)

Python: /home/akash1/scgpt-cna/.venv-scgpt/bin/python
Sample: C3N-00148-01
Matrix: /home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix/matrix.mtx


In [2]:
# load raw matricies

features = pd.read_csv(
    FEATURE_FILE, sep="\t", header=None, dtype=str, na_filter=False
)
barcodes = pd.read_csv(
    BARCODE_FILE, sep="\t", header=None, dtype=str, na_filter=False
)

assert features.shape[1] == 3, features.shape
assert barcodes.shape[1] == 1, barcodes.shape

features.columns = ["gene_id", "gene_symbol", "feature_type"]
assert features["feature_type"].eq("Gene Expression").all()
assert features["gene_id"].is_unique
assert barcodes[0].is_unique
assert features["gene_id"].ne("").all()
assert barcodes[0].ne("").all()

# Stored matrix: genes × cells.
# AnnData convention: cells × genes.
print("Reading sparse counts...", flush=True)
counts = mmread(MATRIX_FILE).T.tocsr()
counts.sum_duplicates()
counts.eliminate_zeros()
counts.sort_indices()

assert sparse.issparse(counts)
assert counts.shape == (len(barcodes), len(features))
assert np.isfinite(counts.data).all()
assert (counts.data >= 0).all()
assert np.allclose(
    counts.data, np.rint(counts.data), rtol=0, atol=1e-6
), "Expected raw integer-valued counts."

obs = pd.DataFrame(
    index=pd.Index(barcodes[0].to_numpy(), name="barcode")
)
obs["case_id"] = CASE_ID
obs["sample_id"] = SAMPLE_ID

var = features.copy()
var.index = pd.Index(var["gene_id"].to_numpy(), name="feature_id")

adata_lab = ad.AnnData(X=counts, obs=obs, var=var)
adata_lab.layers["counts"] = adata_lab.X.copy()

adata_lab.obs["total_counts"] = np.asarray(
    counts.sum(axis=1)
).ravel()
adata_lab.obs["n_genes_detected"] = counts.getnnz(axis=1)

print(adata_lab)
print(f"Nonzero entries: {counts.nnz:,}")
display(adata_lab.var.head())
display(
    adata_lab.obs[["total_counts", "n_genes_detected"]].describe()
)

Reading sparse counts...
AnnData object with n_obs × n_vars = 7598 × 60660
    obs: 'case_id', 'sample_id', 'total_counts', 'n_genes_detected'
    var: 'gene_id', 'gene_symbol', 'feature_type'
    layers: 'counts'
Nonzero entries: 12,599,909


,gene_id,gene_symbol,feature_type
feature_id,,,
ENSG00000223972.5,ENSG00000223972.5,DDX11L1,Gene Expression
ENSG00000227232.5,ENSG00000227232.5,WASH7P,Gene Expression
ENSG00000278267.1,ENSG00000278267.1,MIR6859-1,Gene Expression
ENSG00000243485.5,ENSG00000243485.5,MIR1302-2HG,Gene Expression
ENSG00000284332.1,ENSG00000284332.1,MIR1302-2,Gene Expression


,total_counts,n_genes_detected
count,7598.000000,7598.000000
mean,3166.449329,1658.319163
std,1676.129470,656.013332
min,500.000000,64.000000
25%,2025.500000,1196.250000
50%,2902.000000,1616.000000
75%,3963.750000,2057.000000
max,24279.000000,6980.000000


In [8]:
print(barcodes.head())
print(features.head())


                    0
0  AAACCCAAGAAGTGTT-1
1  AAACCCAAGCCTGTCG-1
2  AAACCCAAGCGCTGCT-1
3  AAACCCAAGTTCCGGC-1
4  AAACCCACAAGCGCTC-1
             gene_id  gene_symbol     feature_type
0  ENSG00000223972.5      DDX11L1  Gene Expression
1  ENSG00000227232.5       WASH7P  Gene Expression
2  ENSG00000278267.1    MIR6859-1  Gene Expression
3  ENSG00000243485.5  MIR1302-2HG  Gene Expression
4  ENSG00000284332.1    MIR1302-2  Gene Expression


In [4]:
# align copyKAT predictions by parcode
copykat = pd.read_csv(COPYKAT_FILE, sep="\t", dtype=str)

required = {"cell.names", "copykat.pred"}
assert required.issubset(copykat.columns), copykat.columns.tolist()
assert copykat["cell.names"].notna().all()
assert copykat["cell.names"].is_unique
assert copykat["copykat.pred"].notna().all()

predictions = copykat.set_index("cell.names")["copykat.pred"]

matched = adata_lab.obs_names.isin(predictions.index)
extra_barcodes = predictions.index.difference(adata_lab.obs_names)

adata_lab.obs["copykat_prediction"] = predictions.reindex(
    adata_lab.obs_names
).to_numpy()

print(f"Matched cells: {matched.sum():,}/{adata_lab.n_obs:,}")
print("CopyKAT barcodes absent from expression:", len(extra_barcodes))
if len(extra_barcodes):
    print("Examples:", extra_barcodes[:5].tolist())

print("\nCopyKAT categories; NaN means no matching output row:")
display(
    adata_lab.obs["copykat_prediction"]
    .value_counts(dropna=False)
    .rename("n_cells")
)

Matched cells: 7,584/7,598
CopyKAT barcodes absent from expression: 0

CopyKAT categories; NaN means no matching output row:


copykat_prediction
aneuploid      4106
diploid        2100
not.defined    1378
NaN              14
Name: n_cells, dtype: int64

In [5]:
vocab = json.loads((MODEL_DIR / "vocab.json").read_text())

symbols = adata_lab.var["gene_symbol"]
in_vocab = symbols.isin(vocab)
keep_for_model = in_vocab & ~symbols.duplicated(keep="first")

adata_lab.var["in_scgpt_vocab"] = in_vocab.to_numpy()
adata_lab.var["keep_for_model"] = keep_for_model.to_numpy()

gene_totals = np.asarray(
    adata_lab.layers["counts"].sum(axis=0)
).ravel()
all_counts = float(gene_totals.sum())
assert all_counts > 0

coverage_rows = []
for label, mask in [
    ("All features", np.ones(adata_lab.n_vars, dtype=bool)),
    ("Vocabulary matches", in_vocab.to_numpy()),
    ("After keeping first symbol occurrence", keep_for_model.to_numpy()),
]:
    retained_counts = float(gene_totals[mask].sum())
    coverage_rows.append({
        "stage": label,
        "n_features": int(mask.sum()),
        "total_counts": retained_counts,
        "percent_counts_retained": 100 * retained_counts / all_counts,
    })

coverage_lab = pd.DataFrame(coverage_rows)
display(coverage_lab.round(3))

print(
    "Repeated symbol occurrences beyond the first:",
    int(symbols.duplicated(keep="first").sum()),
)
print(
    "Matched features removed for duplicate symbols:",
    int((in_vocab & ~keep_for_model).sum()),
)

# Per-cell coverage can reveal losses hidden by the overall average.
retained_per_cell = np.asarray(
    adata_lab.layers["counts"][:, keep_for_model.to_numpy()].sum(axis=1)
).ravel()

totals = adata_lab.obs["total_counts"].to_numpy()
fractions = np.full(adata_lab.n_obs, np.nan)
np.divide(retained_per_cell, totals, out=fractions, where=totals > 0)
adata_lab.obs["model_count_fraction"] = fractions

print("\nPer-cell fraction of counts retained:")
display(adata_lab.obs["model_count_fraction"].describe())

# Identify the most expressed features that fail vocabulary matching.
unmatched = adata_lab.var.loc[
    ~in_vocab, ["gene_id", "gene_symbol"]
].copy()
unmatched["total_counts"] = gene_totals[~in_vocab.to_numpy()]

print("\nMost expressed unmatched features:")
display(
    unmatched.sort_values("total_counts", ascending=False).head(10)
)

,stage,n_features,total_counts,percent_counts_retained
0,All features,60660,24058682.0,100.000
1,Vocabulary matches,39839,22524177.0,93.622
2,After keeping first symbol occurrence,39800,22524114.0,93.622


Repeated symbol occurrences beyond the first: 1233
Matched features removed for duplicate symbols: 39

Per-cell fraction of counts retained:


count    7598.000000
mean        0.936876
std         0.021378
min         0.861687
25%         0.920641
50%         0.933591
75%         0.953378
max         1.000000
Name: model_count_fraction, dtype: float64


Most expressed unmatched features:


,gene_id,gene_symbol,total_counts
feature_id,,,
ENSG00000256947.1,ENSG00000256947.1,AP002518.2,39541
ENSG00000223786.1,ENSG00000223786.1,AL357507.1,20813
ENSG00000287733.1,ENSG00000287733.1,AC083862.2,19851
ENSG00000247199.5,ENSG00000247199.5,AC011346.1,18326
ENSG00000236039.3,ENSG00000236039.3,AC019117.1,12685
ENSG00000272243.6,ENSG00000272243.6,AL356277.3,11009
ENSG00000233005.2,ENSG00000233005.2,AC018742.1,10585
ENSG00000230258.6,ENSG00000230258.6,AC005208.1,10010
ENSG00000275443.2,ENSG00000275443.2,AC012409.2,9726


### 2. Compare lab data to census data

In [ ]:
# load census and compare gene names
CENSUS_FILE = (
    ROOT / "sanity_check/data/census_proximal_tubule_256/reference_raw.h5ad"
)
assert CENSUS_FILE.is_file()

adata_census = ad.read_h5ad(CENSUS_FILE)
print("Lab:", adata_lab.shape)
print("Census:", adata_census.shape)

# Build a lookup using Census's stable Ensembl IDs.
census_annotations = pd.DataFrame({
    "ensembl_base": (
        adata_census.var["feature_id"].astype(str)
        .str.replace(r"\.\d+$", "", regex=True).to_numpy()
    ),
    "census_symbol": (
        adata_census.var["feature_name"].astype(str).to_numpy()
    ),
})

# Exclude ambiguous IDs rather than silently choosing a mapping.
ambiguous = census_annotations["ensembl_base"].duplicated(keep=False)
print("Census rows with ambiguous base IDs:", int(ambiguous.sum()))

symbol_lookup = (
    census_annotations.loc[~ambiguous]
    .set_index("ensembl_base")["census_symbol"]
)

gene_comparison = pd.DataFrame({
    "gene_id": adata_lab.var["gene_id"].to_numpy(),
    "lab_symbol": adata_lab.var["gene_symbol"].astype(str).to_numpy(),
    "total_counts": np.asarray(
        adata_lab.layers["counts"].sum(axis=0)
    ).ravel(),
})
gene_comparison["ensembl_base"] = (
    gene_comparison["gene_id"].str.replace(r"\.\d+$", "", regex=True)
)
gene_comparison["census_symbol"] = (
    gene_comparison["ensembl_base"].map(symbol_lookup)
)

gene_comparison["lab_symbol_in_vocab"] = (
    gene_comparison["lab_symbol"].isin(vocab)
)
gene_comparison["census_symbol_in_vocab"] = (
    gene_comparison["census_symbol"].isin(vocab)
)
gene_comparison["different_symbol"] = (
    gene_comparison["census_symbol"].notna()
    & gene_comparison["lab_symbol"].ne(gene_comparison["census_symbol"])
)

potentially_recovered = (
    ~gene_comparison["lab_symbol_in_vocab"]
    & gene_comparison["census_symbol_in_vocab"]
)

print("Lab features with an unambiguous Census ID match:",
      int(gene_comparison["census_symbol"].notna().sum()))
print("Shared-ID features with different symbols:",
      int(gene_comparison["different_symbol"].sum()))
print("Currently unmatched features with a recognized Census symbol:",
      int(potentially_recovered.sum()))
print(
    "Percent of all lab counts in those potentially recoverable features:",
    round(
        100 * gene_comparison.loc[
            potentially_recovered, "total_counts"
        ].sum() / gene_comparison["total_counts"].sum(),
        3,
    ),
)

display(
    gene_comparison.loc[potentially_recovered]
    .sort_values("total_counts", ascending=False)
    .head(15)
)

Lab: (7598, 60660)
Census: (256, 60664)
Census rows with ambiguous base IDs: 0
Lab features with an unambiguous Census ID match: 60561
Shared-ID features with different symbols: 21170
Currently unmatched features with a recognized Census symbol: 20765
Percent of all lab counts in those potentially recoverable features: 6.363


,gene_id,lab_symbol,total_counts,ensembl_base,census_symbol,lab_symbol_in_vocab,census_symbol_in_vocab,different_symbol
34603,ENSG00000256947.1,AP002518.2,39541,ENSG00000256947,RP11-64D24.2,False,True,True
20178,ENSG00000223786.1,AL357507.1,20813,ENSG00000223786,RP11-554D15.1,False,True,True
24007,ENSG00000287733.1,AC083862.2,19851,ENSG00000287733,RP11-134L10.2,False,True,True
17851,ENSG00000247199.5,AC011346.1,18326,ENSG00000247199,RP11-373N22.3,False,True,True
21868,ENSG00000236039.3,AC019117.1,12685,ENSG00000236039,LINC02889,False,True,True
20180,ENSG00000272243.6,AL356277.3,11009,ENSG00000272243,RP11-554D15.3,False,True,True
5799,ENSG00000233005.2,AC018742.1,10585,ENSG00000233005,AC067959.1,False,True,True
49206,ENSG00000230258.6,AC005208.1,10010,ENSG00000230258,AC002539.1,False,True,True
43912,ENSG00000275443.2,AC012409.2,9726,ENSG00000275443,RP11-759A24.3,False,True,True
59782,ENSG00000284377.1,AL589669.1,9471,ENSG00000284377,RP11-226B15.1,False,True,True


In [ ]:
# load historical preprocessing function
HISTORICAL = (
    ROOT / "sanity_check/reference_4068d67/scgpt/preprocess.py"
)
assert HISTORICAL.is_file()

import importlib.util

spec = importlib.util.spec_from_file_location(
    "transfer_historical_preprocess", HISTORICAL
)
historical = importlib.util.module_from_spec(spec)
spec.loader.exec_module(historical)
assert historical._digitize.__defaults__ == ("one",)

config = json.loads((MODEL_DIR / "args.json").read_text())
assert config["n_bins"] == 51

def profile_dataset(adata, symbol_column):
    raw = sparse.csr_matrix(adata.layers["counts"], copy=True)
    raw.sum_duplicates()
    raw.eliminate_zeros()

    names = adata.var[symbol_column].astype(str)
    keep = names.isin(vocab) & ~names.duplicated(keep="first")
    selected = raw[:, keep.to_numpy()].tocsr()

    total = np.asarray(raw.sum(axis=1)).ravel()
    detected = raw.getnnz(axis=1)
    model_total = np.asarray(selected.sum(axis=1)).ravel()
    model_detected = selected.getnnz(axis=1)

    assert (total > 0).all()
    assert (detected > 0).all()
    assert (model_detected > 0).all()

    # Count genes with exactly one detected molecule/count.
    ones = raw.copy()
    ones.data = (ones.data == 1).astype(np.int32)
    n_ones = np.asarray(ones.sum(axis=1)).ravel()

    records = []
    for i in range(adata.n_obs):
        values = selected.getrow(i).data.astype(np.float32)
        binned = historical.binning(values.copy(), n_bins=51)
        labels, frequencies = np.unique(binned, return_counts=True)
        count_one = values == 1

        records.append({
            "model_fraction_count_one": float(count_one.mean()),
            "count_one_bin": (
                int(binned[count_one][0]) if count_one.any() else np.nan
            ),
            "occupied_bins": len(labels),
            "dominant_bin_fraction": float(frequencies.max() / len(binned)),
        })

        if (i + 1) % 2000 == 0:
            print(f"Profiled {i + 1:,}/{adata.n_obs:,} cells", flush=True)

    profile = pd.DataFrame(records, index=adata.obs_names)
    profile["total_counts"] = total
    profile["detected_genes"] = detected
    profile["model_detected_genes"] = model_detected
    profile["model_count_fraction"] = model_total / total
    profile["raw_fraction_count_one"] = n_ones / detected

    return profile, selected, keep

lab_profile, lab_model_counts, lab_keep = profile_dataset(
    adata_lab, "gene_symbol"
)
census_profile, census_model_counts, census_keep = profile_dataset(
    adata_census, "feature_name"
)

# Carry annotations alongside the lab measurements.
lab_profile["copykat_prediction"] = adata_lab.obs["copykat_prediction"]

metrics = [
    "total_counts",
    "detected_genes",
    "model_detected_genes",
    "model_count_fraction",
    "raw_fraction_count_one",
    "model_fraction_count_one",
    "count_one_bin",
    "occupied_bins",
    "dominant_bin_fraction",
]

comparison_rows = []
for metric in metrics:
    row = {"metric": metric}
    for name, profile in [
        ("lab", lab_profile), ("census", census_profile)
    ]:
        values = profile[metric].dropna()
        row[f"{name}_q25"] = values.quantile(0.25)
        row[f"{name}_median"] = values.median()
        row[f"{name}_q75"] = values.quantile(0.75)
    comparison_rows.append(row)

distribution_comparison = pd.DataFrame(comparison_rows).set_index("metric")
display(distribution_comparison.round(3))

for name, profile in [("Lab", lab_profile), ("Census", census_profile)]:
    print(
        f"{name}: "
        f"{100 * profile['model_detected_genes'].gt(1199).mean():.1f}% "
        "of cells exceed 1,199 matched detected genes."
    )

/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


Profiled 2,000/7,598 cells
Profiled 4,000/7,598 cells
Profiled 6,000/7,598 cells


,lab_q25,lab_median,lab_q75,census_q25,census_median,census_q75
metric,,,,,,
total_counts,2025.500,2902.000,3963.750,772.000,1571.500,3399.750
detected_genes,1196.250,1616.000,2057.000,472.750,786.500,1294.250
model_detected_genes,1099.000,1481.000,1874.000,472.750,786.500,1294.250
model_count_fraction,0.921,0.934,0.953,1.000,1.000,1.000
raw_fraction_count_one,0.705,0.747,0.790,0.645,0.697,0.767
model_fraction_count_one,0.700,0.744,0.788,0.645,0.697,0.767
count_one_bin,35.000,37.000,39.000,32.000,35.000,38.000
occupied_bins,5.000,6.000,6.000,6.000,7.000,9.000
dominant_bin_fraction,0.700,0.744,0.788,0.645,0.697,0.767


Lab: 68.4% of cells exceed 1,199 matched detected genes.
Census: 30.1% of cells exceed 1,199 matched detected genes.


In [11]:
grouped = lab_profile.copy()
grouped["copykat_group"] = grouped["copykat_prediction"].fillna(
    "missing_from_output"
)

group_summary = grouped.groupby("copykat_group").agg(
    n_cells=("total_counts", "size"),
    median_counts=("total_counts", "median"),
    median_detected_genes=("detected_genes", "median"),
    median_model_detected_genes=("model_detected_genes", "median"),
    median_count_coverage=("model_count_fraction", "median"),
    median_fraction_count_one=("model_fraction_count_one", "median"),
    median_count_one_bin=("count_one_bin", "median"),
    median_occupied_bins=("occupied_bins", "median"),
)

display(group_summary.round(3))

,n_cells,median_counts,median_detected_genes,median_model_detected_genes,median_count_coverage,median_fraction_count_one,median_count_one_bin,median_occupied_bins
copykat_group,,,,,,,,
aneuploid,4106,3367.0,1818.0,1647.0,0.924,0.735,37.0,6.0
diploid,2100,2916.0,1590.0,1495.5,0.956,0.725,36.0,6.0
missing_from_output,14,1556.5,136.5,135.5,0.999,0.798,39.5,8.5
not.defined,1378,1456.0,876.5,819.0,0.950,0.812,40.0,5.0


In [12]:
# Confirm gene_comparison is still in the original feature order.
assert np.array_equal(
    gene_comparison["gene_id"].to_numpy(),
    adata_lab.var["gene_id"].to_numpy(),
)

original_symbols = adata_lab.var["gene_symbol"].astype(str)

census_symbols = pd.Series(
    gene_comparison["census_symbol"].to_numpy(),
    index=adata_lab.var_names,
)

# Prefer the ID-matched Census symbol when it is recognized.
# Otherwise retain the original lab symbol.
use_census = census_symbols.isin(vocab)
model_symbols = original_symbols.where(~use_census, census_symbols)

adata_lab.var["model_symbol"] = model_symbols
adata_lab.var["model_symbol_source"] = np.where(
    use_census, "Ensembl-ID-matched Census annotation", "original lab annotation"
)

new_in_vocab = model_symbols.isin(vocab)
new_keep = new_in_vocab & ~model_symbols.duplicated(keep="first")

adata_lab.var["keep_for_model_harmonized"] = new_keep.to_numpy()

gene_totals = np.asarray(
    adata_lab.layers["counts"].sum(axis=0)
).ravel()

rows = []
for name, mask in [
    ("Original matching, after duplicate handling", lab_keep.to_numpy()),
    ("Harmonized matching, before duplicate handling", new_in_vocab.to_numpy()),
    ("Harmonized matching, after duplicate handling", new_keep.to_numpy()),
]:
    rows.append({
        "selection": name,
        "n_features": int(mask.sum()),
        "percent_counts_retained": (
            100 * gene_totals[mask].sum() / gene_totals.sum()
        ),
    })

display(pd.DataFrame(rows).round(4))

# Inspect all recognized features involved in a repeated model symbol.
duplicate_mask = (
    new_in_vocab & model_symbols.duplicated(keep=False)
)
duplicates = adata_lab.var.loc[
    duplicate_mask, ["gene_id", "gene_symbol", "model_symbol"]
].copy()
duplicates["total_counts"] = gene_totals[duplicate_mask.to_numpy()]

print("Recognized features involved in duplicate model symbols:",
      len(duplicates))
print("Counts discarded by keeping the first occurrence:",
      float(gene_totals[(new_in_vocab & ~new_keep).to_numpy()].sum()))

display(
    duplicates.sort_values(
        ["model_symbol", "total_counts"], ascending=[True, False]
    ).head(30)
)

,selection,n_features,percent_counts_retained
0,"Original matching, after duplicate handling",39800,93.6216
1,"Harmonized matching, before duplicate handling",60604,99.9850
2,"Harmonized matching, after duplicate handling",60566,99.9848


Recognized features involved in duplicate model symbols: 76
Counts discarded by keeping the first occurrence: 63.0


,gene_id,gene_symbol,model_symbol,total_counts
feature_id,,,,
ENSG00000182584.6,ENSG00000182584.6,ACTL10,ACTL10,1
ENSG00000288649.1,ENSG00000288649.1,ACTL10,ACTL10,0
ENSG00000197976.12,ENSG00000197976.12,AKAP17A,AKAP17A,657
ENSG00000197976.12_PAR_Y,ENSG00000197976.12_PAR_Y,AKAP17A,AKAP17A,0
ENSG00000237801.6,ENSG00000237801.6,AMD1P2,AMD1P2,1
ENSG00000237801.6_PAR_Y,ENSG00000237801.6_PAR_Y,AMD1P2,AMD1P2,0
ENSG00000196433.13,ENSG00000196433.13,ASMT,ASMT,10
ENSG00000196433.13_PAR_Y,ENSG00000196433.13_PAR_Y,ASMT,ASMT,0
ENSG00000169093.16,ENSG00000169093.16,ASMTL,ASMTL,414


In [13]:
# Preserve the original matching results for comparison.
lab_profile_original = lab_profile.copy()
lab_model_counts_original = lab_model_counts

# Reuse our profiling function, now matching through harmonized symbols.
(
    lab_profile_harmonized,
    lab_model_counts_harmonized,
    lab_keep_harmonized,
) = profile_dataset(adata_lab, "model_symbol")

lab_profile_harmonized["copykat_prediction"] = (
    adata_lab.obs["copykat_prediction"]
)

assert np.array_equal(
    lab_keep_harmonized.to_numpy(),
    adata_lab.var["keep_for_model_harmonized"].to_numpy(),
)

# Integer vocabulary IDs, aligned with the model matrix's columns.
lab_gene_ids = np.array([
    vocab[s]
    for s in adata_lab.var.loc[lab_keep_harmonized, "model_symbol"]
], dtype=np.int64)

assert lab_model_counts_harmonized.shape == (
    adata_lab.n_obs, len(lab_gene_ids)
)
assert len(np.unique(lab_gene_ids)) == len(lab_gene_ids)

profiles = {
    "Lab: original names": lab_profile_original,
    "Lab: harmonized names": lab_profile_harmonized,
    "Census": census_profile,
}

comparison_metrics = [
    "model_detected_genes",
    "model_count_fraction",
    "model_fraction_count_one",
    "count_one_bin",
    "occupied_bins",
    "dominant_bin_fraction",
]

median_comparison = pd.DataFrame({
    name: profile[comparison_metrics].median()
    for name, profile in profiles.items()
})
display(median_comparison.round(4))

print("Final lab model matrix:", lab_model_counts_harmonized.shape)
print(
    "Harmonized lab cells exceeding 1,199 detected model genes:",
    f"{100 * lab_profile_harmonized['model_detected_genes'].gt(1199).mean():.1f}%",
)

Profiled 2,000/7,598 cells
Profiled 4,000/7,598 cells
Profiled 6,000/7,598 cells


,Lab: original names,Lab: harmonized names,Census
model_detected_genes,1481.0000,1616.0000,786.5000
model_count_fraction,0.9336,1.0000,1.0000
model_fraction_count_one,0.7437,0.7473,0.6968
count_one_bin,37.0000,37.0000,35.0000
occupied_bins,6.0000,6.0000,7.0000
dominant_bin_fraction,0.7437,0.7473,0.6968


Final lab model matrix: (7598, 60566)
Harmonized lab cells exceeding 1,199 detected model genes: 74.7%


### 3. Run small execution check

### 4. Evaluate transferability for this sample (kidney case)

### 5. Evaluate transferability for all samples in the lab data